# Áreas de risco alimentar e vulnerabilidade social no Brasil

**Estudo ecológico municipal — sobreposição de critérios.**

**Pergunta da pesquisa:** identificar áreas do Brasil de maior risco alimentar
e vulnerabilidade social a partir de IVS, IDHM, CadÚnico, CadInsan e SISVAN.

**Objetivo da análise:** identificar municípios onde coincidem IVS elevado,
IDHM baixo, risco alimentar estimado elevado no CadInsan e déficit de
altura para idade elevado nas crianças acompanhadas pelo SISVAN.
CadÚnico ajuda a caracterizar esses territórios.

A unidade de análise é o **município**, não a pessoa ou a família. A
sobreposição identifica convergência de indicadores, sem ranking composto,
inferência causal ou estimativa de fome em toda a população municipal.

## 1. Bases de dados e procedência

A análise reúne **quatro arquivos municipais**, que contêm os cinco
indicadores da pesquisa. IVS e IDHM estão no mesmo arquivo.
IVS significa **Índice de Vulnerabilidade Social**; IDHM, **Índice de
Desenvolvimento Humano Municipal**; CadÚnico, **Cadastro Único para
Programas Sociais**. CadInsan é o indicador municipalizado de risco de
insegurança alimentar grave a partir do CadÚnico.

| Base | Fonte original / instituição responsável | De onde o arquivo foi obtido | Referência e unidade | Registros municipais |
|---|---|---|---|---:|
| IVS e IDHM | [Atlas IVS / Ipea](http://ivs.ipea.gov.br/index.php/pt/planilha); IDHM do Atlas do Desenvolvimento Humano (PNUD, Ipea e FJP) | [CSV publicado por Cozinhas Solidárias](https://raw.githubusercontent.com/TriangulosTecnologia/cozsolidarias/refs/heads/main/src/data-source-static/data/atlasivs_municipios_2010.csv) | 2010; índices municipais | 5.565 |
| CadÚnico | [MDS / SAGI](https://aplicacoes.mds.gov.br/sagi/servicos/misocial) | [JSON publicado por Cozinhas Solidárias](https://raw.githubusercontent.com/TriangulosTecnologia/cozsolidarias/refs/heads/main/public/geo/municipios-cadunico.json) | Junho/2026; pessoas cadastradas | 5.564 |
| CadInsan | [MDS — relatório CadInsan 2025](https://www.gov.br/mds/pt-br/Sisan/vigilancia-do-sisan/CADINSAN2025.pdf) | [CSV publicado por Cozinhas Solidárias](https://raw.githubusercontent.com/TriangulosTecnologia/cozsolidarias/refs/heads/main/src/data-source-static/data/CADINSAN_2025_dados_municipais.csv) | Janeiro/2025; famílias do universo analisado | 5.570 |
| SISVAN | Ministério da Saúde — Sistema de Vigilância Alimentar e Nutricional | Coleta direta dos [relatórios públicos](https://sisaps.saude.gov.br/sisvan/relatoriopublico/) de Altura X Idade | 2025; crianças de 0 a menos de 5 anos acompanhadas pelo sistema | 5,571 |

**Procedência:** as três bases sociais foram obtidas do
[repositório de Cozinhas Solidárias](https://github.com/TriangulosTecnologia/cozsolidarias),
não extraídas diretamente dos portais oficiais nesta pesquisa. Seus períodos
e unidades foram conferidos no catálogo que acompanha os arquivos. O
relatório oficial do MDS complementa a referência mensal do CadInsan.
SISVAN foi baixado novamente em **2026-10-04**, nas 27 UFs,
usando o coletor atualizado. Os XLSX oficiais são preservados sem alteração.
O CSV consolida as linhas municipais e achata cabeçalhos multinível; mantém
os valores de contagem e os percentuais das células, sem DAI pré-calculado.
**DAI é calculado nesta análise**, não na base de entrada.

**Apoio cartográfico:** malha municipal simplificada obtida diretamente da
[API v4 do IBGE](https://servicodados.ibge.gov.br/api/docs/malhas?versao=4).
A API utilizada não informa o ano da malha; ela é apoio ilustrativo, não uma
harmonização dos limites municipais entre os diferentes períodos.

Os arquivos estão incorporados ao notebook. Para reproduzir no Colab,
revisar os parâmetros e selecionar **Ambiente de execução → Executar tudo**.
A preparação do ambiente pode exigir internet; a análise não faz nova coleta.

## 2. Estratégia e parâmetros da análise

A regra principal exige **os quatro critérios simultaneamente**: IVS,
CadInsan e DAI elevados, e IDHM baixo. CadÚnico contextualiza os
resultados, sem participar da seleção principal.

IVS ≥ **0,401** e IDHM < **0,600** são cortes fixos, correspondentes a
vulnerabilidade alta/muito alta e desenvolvimento baixo/muito baixo.
Não mudam com `QUANTIL` nem na análise de sensibilidade. Os valores originais
são preservados. IDHM igual a 0,600 não satisfaz o critério.

P75 para CadInsan/DAI e mínimo de 100 avaliações são pontos de partida
exploratórios, editáveis e sem caráter oficial. O corte do DAI é calculado
sobre municípios com o denominador mínimo; CadInsan usa seu universo válido.
Esses quantis são nacionais, não calculados apenas sobre municípios que
atendem aos cortes sociais.
O percentil 75 delimita aproximadamente os 25% maiores valores municipais,
não 75% da população. Cada município tem o mesmo peso; empates são incluídos.
O mínimo de 100 avaliações é uma precaução operacional, não uma exigência
oficial nem garantia de representatividade; será examinado na sensibilidade.

`com_PBF` é o cenário inicial, considerando o efeito do benefício na renda.
`sem_PBF` é o cenário contrafactual que desconsidera esse efeito e entra na
sensibilidade. Não são dois grupos de beneficiários e não beneficiários.
CadInsan considera famílias com cadastro atualizado nos últimos 12 meses,
com referência janeiro/2025. As pessoas cadastradas no JSON de junho/2026
não substituem esse denominador de famílias.

In [1]:
QUANTIL = 0.75  # CadInsan e DAI
MINIMO_AVALIADOS = 100
CENARIO_CADINSAN = "com_PBF"  # opções: "com_PBF", "sem_PBF"
QUANTIS_SENSIBILIDADE = [0.75, 0.80]
MINIMOS_SENSIBILIDADE = [30, 50, 100]
CENARIOS_SENSIBILIDADE = ["com_PBF", "sem_PBF"]
GERAR_MAPAS = True
BAIXAR_RESULTADOS_NO_COLAB = False  # True solicita download do ZIP no final

### Preparação da execução

As células técnicas recolhidas preparam o ambiente e recuperam os arquivos.
A integridade das entradas é verificada antes da análise; os arquivos
originais permanecem separados dos resultados derivados.

In [2]:
import importlib.util
import subprocess
import sys
from pathlib import Path

packages = {"pandas": "pandas>=2.2,<4", "numpy": "numpy>=1.26,<3",
            "matplotlib": "matplotlib>=3.8,<4", "scipy": "scipy>=1.11,<2"}
missing = [requirement for module, requirement in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import hashlib
import io
import json
import zipfile
import base64
import platform
import shutil
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import scipy
from IPython.display import display, Markdown

PASTA_EXECUCAO = Path.cwd() / "resultados_sobreposicao"
PASTA_DADOS = PASTA_EXECUCAO / "entradas"
PASTA_SAIDA = PASTA_EXECUCAO / "resultados"
PASTA_FIGURAS = PASTA_SAIDA / "figuras"
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
PASTA_FIGURAS.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")
CODIGO_ANALISE_SHA256 = "54148597afd403b239e162d95df24a8cd59d7c1cecbb4878401afae5b41cc7f9"

In [3]:
# @title Bases e malha incorporadas — executar sem editar
PACOTE_BASE64 = "..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PACOTE_BASE64))) as pacote:
    for membro in pacote.infolist():
        destino = (PASTA_DADOS / membro.filename).resolve()
        if not destino.is_relative_to(PASTA_DADOS.resolve()):
            raise ValueError("Caminho inválido no pacote de dados")
    pacote.extractall(PASTA_DADOS)
print("Bases recuperadas para análise.")

Bases recuperadas para análise.


## 3. Integração e disponibilidade dos dados

A união das bases preserva todos os municípios disponíveis. A integração
verifica a unicidade dos códigos, categorias e denominadores. Ausência de
informação é mantida como ausência, não substituída por zero.

In [4]:
# @title Funções de análise — executar sem editar
"""Análise municipal exploratória usada pelo notebook autocontido do Colab."""

from __future__ import annotations

import hashlib
import json
from decimal import Decimal, InvalidOperation
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd


DATA_FILES = {
    "ivs_idhm": "ivs_idhm/atlasivs_municipios_2010.csv",
    "cadunico": "cadunico/municipios-cadunico.json",
    "cadinsan": "cadinsan/CADINSAN_2025_dados_municipais.csv",
    "sisvan": "sisvan/sisvan_municipios_altura_por_idade_0_a_menor_5_anos_2025.csv",
}
CATALOGUE_DATASETS = {
    "ivs_idhm": "municipios_ivs",
    "cadunico": "municipios_cadunico",
    "cadinsan": "municipios_cadinsan",
}
FIXED_CUTS = {"ivs": (0.401, ">="), "idhm": (0.600, "<")}
GROUP_COLORS = {
    "Convergência dos quatro critérios": "#9e1b32",
    "IVS e CadInsan elevados, IDHM baixo, DAI abaixo do corte": "#e89c38",
    "DAI elevado sem convergência social-alimentar": "#377eb8",
    "Outras combinações": "#d7e4df",
    "Informação insuficiente": "#b9b9b9",
}
UF_REGIONS = {
    "11": ("RO", "Norte"), "12": ("AC", "Norte"),
    "13": ("AM", "Norte"), "14": ("RR", "Norte"),
    "15": ("PA", "Norte"), "16": ("AP", "Norte"),
    "17": ("TO", "Norte"), "21": ("MA", "Nordeste"),
    "22": ("PI", "Nordeste"), "23": ("CE", "Nordeste"),
    "24": ("RN", "Nordeste"), "25": ("PB", "Nordeste"),
    "26": ("PE", "Nordeste"), "27": ("AL", "Nordeste"),
    "28": ("SE", "Nordeste"), "29": ("BA", "Nordeste"),
    "31": ("MG", "Sudeste"), "32": ("ES", "Sudeste"),
    "33": ("RJ", "Sudeste"), "35": ("SP", "Sudeste"),
    "41": ("PR", "Sul"), "42": ("SC", "Sul"), "43": ("RS", "Sul"),
    "50": ("MS", "Centro-Oeste"), "51": ("MT", "Centro-Oeste"),
    "52": ("GO", "Centro-Oeste"), "53": ("DF", "Centro-Oeste"),
}


def verify_hashes(directory):
    directory = Path(directory)
    records = []
    for line in (directory / "SHA256SUMS").read_text().splitlines():
        if not line.strip():
            continue
        expected, relative = line.split(maxsplit=1)
        relative = relative.lstrip("*")
        path = directory / relative
        actual = hashlib.sha256(path.read_bytes()).hexdigest()
        if actual != expected:
            raise ValueError(f"Integridade inválida: {relative}")
        records.append({"arquivo": relative, "sha256": actual, "integridade": "OK"})
    if {r["arquivo"] for r in records} != set(DATA_FILES.values()):
        raise ValueError("O manifesto deve identificar exatamente as quatro bases.")
    return pd.DataFrame(records)


def validate_catalogue(catalogue_path, directory):
    """Confere que os metadados sociais descrevem exatamente os arquivos usados."""
    catalogue_path, directory = Path(catalogue_path), Path(directory)
    raw = catalogue_path.read_bytes()
    catalogue = json.loads(raw.decode("utf-8-sig"))
    records = []
    for source, dataset_id in CATALOGUE_DATASETS.items():
        dataset = catalogue["datasets"][dataset_id]
        relative = DATA_FILES[source]
        actual = hashlib.sha256((directory / relative).read_bytes()).hexdigest()
        expected = dataset["stats"]["checkSum"].removeprefix("sha256:")
        if actual != expected:
            raise ValueError(f"Catálogo não corresponde ao arquivo {relative}.")
        temporal = dataset["temporal"]["extent"][0]
        fields = dataset["schema"]["fields"]
        units = sorted({f["unit"] for f in fields if "unit" in f})
        records.append({"dataset_id": dataset_id, "arquivo": relative,
                        "titulo": dataset["title"], "inicio_referencia_catalogo": temporal[0],
                        "fim_referencia_catalogo": temporal[1], "unidades_catalogo": "; ".join(units),
                        "fonte_original": dataset["source"]["url"],
                        "sha256": actual, "corresponde_ao_catalogo": True})
    provenance = {"catalogo_id": catalogue["catalog"]["id"],
                  "catalogo_status": catalogue["catalog"]["status"],
                  "catalogo_atualizado_em": catalogue["catalog"]["updated_at"],
                  "catalogo_sha256": hashlib.sha256(raw).hexdigest(),
                  "schema_version": catalogue["schema_version"],
                  "fontes": records,
                  "cadunico": {"unidade": "pessoas", "periodo": "2026-06",
                               "campo_fonte": "cadun_qtd_pessoas_cadastradas_i",
                               "evidencia": "datasets.municipios_cadunico.source.notes e schema.fields"},
                  "cadinsan": {"unidade": "famílias", "referencia_catalogo": "2025",
                               "referencia_relatorio_oficial": "2025-01",
                               "populacao": "Famílias do universo analisado pelo CadInsan; não todas as pessoas cadastradas",
                               "cenarios": {"com_PBF": "Risco estimado considerando o efeito do PBF na renda",
                                            "sem_PBF": "Cenário contrafactual desconsiderando o efeito do PBF"},
                               "relatorio_oficial": "https://www.gov.br/mds/pt-br/Sisan/vigilancia-do-sisan/CADINSAN2025.pdf",
                               "evidencia": "Relatório: metodologia, p. 7; cenários, p. 16–19; tabela municipal, p. 22. "
                                            "Valores municipais conferidos por amostragem com o CSV."}}
    return pd.DataFrame(records), provenance


def numeric(values, label):
    """Lê ponto/vírgula decimal e %, sem converter ausência em zero."""
    clean = values.astype("string").str.strip()
    clean = clean.str.replace("%", "", regex=False).str.replace(",", ".", regex=False)
    present = clean.notna() & clean.ne("")
    result = pd.to_numeric(clean.mask(~present), errors="coerce").astype(float)
    invalid = present & result.isna()
    if invalid.any():
        raise ValueError(f"Valores não numéricos em {label}: {values[invalid].head().tolist()}")
    if not np.isfinite(result.dropna()).all():
        raise ValueError(f"Valores infinitos em {label}")
    return result


def codes(values, length, source):
    result = values.astype("string").str.strip()
    if not result.str.fullmatch(rf"\d{{{length}}}").fillna(False).all():
        raise ValueError(f"{source}: códigos devem ter {length} dígitos.")
    if result.duplicated().any():
        raise ValueError(f"{source}: código municipal duplicado.")
    return result


def check_range(values, low, high, label):
    if not values.dropna().between(low, high).all():
        raise ValueError(f"{label}: valores fora de [{low}, {high}].")


def read_sources(directory):
    directory = Path(directory)
    hashes = verify_hashes(directory)
    sources = {}
    for key, relative in DATA_FILES.items():
        if key == "cadunico":
            def unique_keys(pairs):
                result = {}
                for code, value in pairs:
                    if code in result:
                        raise ValueError(f"CadÚnico: chave JSON duplicada: {code}")
                    result[code] = value
                return result
            data = json.loads((directory / relative).read_text(encoding="utf-8"),
                              object_pairs_hook=unique_keys)
            sources[key] = pd.DataFrame({"codigo_ibge_7": list(data),
                                         "cadunico_valor_original": list(data.values())})
        else:
            sources[key] = pd.read_csv(directory / relative, dtype="string",
                                      encoding="utf-8-sig", keep_default_na=False)
    return sources, hashes


def sisvan_metadata(directory):
    path = Path(directory) / DATA_FILES["sisvan"]
    metadata = json.loads(path.with_suffix(".metadados.json").read_text(encoding="utf-8"))
    expected = {"ano": 2025, "fase": "crianca", "indice": "altura_por_idade",
                "faixa_etaria": "0_a_menor_5_anos", "nu_idade_inicio": "0", "nu_idade_fim": "5"}
    if any(metadata.get(key) != value for key, value in expected.items()):
        raise ValueError("SISVAN fora do recorte previsto: altura, 2025, menores de 5 anos.")
    if metadata.get("indicadores_derivados") != []:
        raise ValueError("A entrada SISVAN deve conter somente valores da fonte.")
    if metadata.get("sha256_csv") != hashlib.sha256(path.read_bytes()).hexdigest():
        raise ValueError("Metadados SISVAN não correspondem ao CSV.")
    if set(metadata.get("ufs", [])) != {uf for uf, _ in UF_REGIONS.values()}:
        raise ValueError("SISVAN: coleta nacional incompleta.")
    return metadata


def interpret_sisvan_counts(values, percentages, code):
    """Interpreta artefatos numéricos sem modificar as células de entrada.

    Inteiros admitem a escala original ou ×1000; decimais do XLSX são
    interpretados como milhares. Exige soma exata e percentuais a 0,011 pp.
    Quando mais de uma escala é possível, conserva o menor total coerente,
    explicitamente sinalizando a ambiguidade para auditoria.
    """
    candidates = []
    for raw in values:
        text = str(raw).strip()
        try:
            value = Decimal(text)
        except InvalidOperation as exc:
            raise ValueError(f"{code}: contagem SISVAN não numérica: {raw!r}") from exc
        if not value.is_finite() or value < 0:
            raise ValueError(f"{code}: contagem SISVAN inválida: {raw!r}")
        if value == value.to_integral_value():
            integer = int(value)
            candidates.append([integer, integer * 1000] if integer else [0])
        else:
            scaled = value * 1000
            if scaled != scaled.to_integral_value():
                raise ValueError(f"{code}: escala decimal SISVAN irresolvível: {raw!r}")
            candidates.append([int(scaled)])
    pcts = []
    for raw in percentages:
        text = str(raw).strip()
        pct = 0.0 if text in {"-", "–", "—"} else float(text.rstrip("%").replace(",", "."))
        if not np.isfinite(pct) or not 0 <= pct <= 100:
            raise ValueError(f"{code}: percentual SISVAN inválido: {raw!r}")
        pcts.append(pct)
    solutions = []
    for counts in product(*candidates[:-1]):
        total = sum(counts)
        if total not in candidates[-1]:
            continue
        if total == 0:
            if any(pcts):
                continue
        elif not all(abs(n / total * 100 - pct) <= 0.011 for n, pct in zip(counts, pcts)):
            continue
        solutions.append((*counts, total))
    if not solutions:
        raise ValueError(f"{code}: contagens não conciliam soma e percentuais oficiais.")
    chosen = min(solutions, key=lambda item: item[-1])
    changed = any(Decimal(str(raw)) != n for raw, n in zip(values, chosen))
    return chosen, changed, len(solutions)


def prepare_base(directory):
    """Normaliza cópias em memória e integra por prefixos únicos, com auditoria."""
    sources, hashes = read_sources(directory)
    tables = {}
    ivs = sources["ivs_idhm"]
    tables["ivs_idhm"] = pd.DataFrame({
        "codigo_ivs_original": codes(ivs["municipio"], 7, "IVS/IDHM"),
        "municipio_ivs": ivs["nome_municipio_uf"],
    })
    for column in ivs.columns:
        if column not in ("municipio", "nome_municipio_uf"):
            tables["ivs_idhm"][column] = numeric(ivs[column], column)
            check_range(tables["ivs_idhm"][column], 0, 1, column)

    cad = sources["cadunico"]
    tables["cadunico"] = pd.DataFrame({
        "codigo_cadunico_original": codes(cad["codigo_ibge_7"], 7, "CadÚnico"),
        "cadunico_valor_original": numeric(cad["cadunico_valor_original"], "CadÚnico"),
    })
    ci = sources["cadinsan"]
    tables["cadinsan"] = pd.DataFrame({
        "codigo_cadinsan_original": codes(ci["Cod_IBGE"], 7, "CadInsan"),
        "municipio_cadinsan": ci["Município"],
    })
    renames = {
        "Cadastros_Cadunico": "cadastros_cadunico_cadinsan",
        "Cadinsan_absoluto_com_PBF": "cadinsan_n_com_PBF",
        "Cadinsan_absoluto_sem_PBF": "cadinsan_n_sem_PBF",
        "Cadinsan_proporcional_com_PBF": "cadinsan_pct_com_PBF",
        "Cadinsan_proporcional_sem_PBF": "cadinsan_pct_sem_PBF",
    }
    for original, new in renames.items():
        tables["cadinsan"][new] = numeric(ci[original], original)
    for scenario in ("com_PBF", "sem_PBF"):
        tables["cadinsan"].rename(
            columns={f"cadinsan_pct_{scenario}": f"cadinsan_pct_{scenario}_arquivo"}, inplace=True)
    sv = sources["sisvan"]
    metadata = sisvan_metadata(directory)
    if list(sv.columns) != metadata["colunas"] or len(sv) != metadata["linhas"]:
        raise ValueError("SISVAN: esquema ou número de linhas difere dos metadados.")
    tables["sisvan"] = pd.DataFrame({
        "codigo_sisvan_original": codes(sv["Código IBGE"], 6, "SISVAN"),
        "municipio_sisvan": sv["Município"], "uf_sisvan": sv["UF"],
        "ano_sisvan": metadata["ano"],
    })
    counts = {
        "Altura Muito Baixa para a Idade - Quantidade": "altura_muito_baixa_n",
        "Altura Baixa para a Idade - Quantidade": "altura_baixa_n",
        "Altura Adequada para a Idade - Quantidade": "altura_adequada_n",
        "Total": "avaliados_altura",
    }
    percentages = [original.replace(" - Quantidade", " - %") for original in list(counts)[:3]]
    interpreted = [interpret_sisvan_counts(
        [row[c] for c in counts], [row[c] for c in percentages], row["Código IBGE"])
        for _, row in sv.iterrows()]
    for original, new in counts.items():
        tables["sisvan"][new + "_valor_fonte"] = sv[original]
    for index, new in enumerate(counts.values()):
        tables["sisvan"][new] = [item[0][index] for item in interpreted]
    for original, new in zip(percentages, list(counts.values())[:3]):
        tables["sisvan"][new + "_percentual_fonte"] = sv[original]
    tables["sisvan"]["sisvan_escala_alterada"] = [item[1] for item in interpreted]
    tables["sisvan"]["sisvan_escalas_compativeis"] = [item[2] for item in interpreted]
    for source, table in tables.items():
        original = next(c for c in table if c.startswith("codigo_"))
        table["codigo_ibge_6"] = table[original].str[:6]
        if table["codigo_ibge_6"].duplicated().any():
            raise ValueError(f"{source}: prefixos municipais não são únicos.")
        table[f"tem_{source}"] = True

    # União externa: ausência de uma fonte não elimina o território das demais.
    base = tables["sisvan"]
    for source in ("ivs_idhm", "cadunico", "cadinsan"):
        base = base.merge(tables[source], on="codigo_ibge_6", how="outer",
                          validate="one_to_one")
    for source in tables:
        base[f"tem_{source}"] = base[f"tem_{source}"].fillna(False).astype(bool)
    original_7 = ["codigo_cadinsan_original", "codigo_ivs_original", "codigo_cadunico_original"]
    base["codigo_ibge_7"] = base[original_7].bfill(axis=1).iloc[:, 0]
    for column in original_7:
        conflict = base[column].notna() & base["codigo_ibge_7"].ne(base[column])
        if conflict.any():
            raise ValueError("Conflito entre códigos de sete dígitos das fontes.")
    # O código de 7 dígitos desconhecido permanece ausente; nunca inventar DV.
    base["municipio"] = base[["municipio_sisvan", "municipio_cadinsan", "municipio_ivs"]].bfill(axis=1).iloc[:, 0]
    base["cadunico_pessoas_2026_06"] = base["cadunico_valor_original"]
    base["cadunico_referencia"] = "2026-06"
    base["cadinsan_referencia"] = "2025-01"
    base["uf"] = base["codigo_ibge_6"].str[:2].map(lambda c: UF_REGIONS[c][0])
    base["regiao"] = base["codigo_ibge_6"].str[:2].map(lambda c: UF_REGIONS[c][1])
    sv_rows = base["tem_sisvan"]
    if base.loc[sv_rows, "uf"].ne(base.loc[sv_rows, "uf_sisvan"]).any():
        raise ValueError("UF SISVAN incompatível com o prefixo municipal.")

    nonnegative = [c for c in base if c.endswith("_n") or c == "avaliados_altura"]
    nonnegative += ["cadunico_valor_original", "cadastros_cadunico_cadinsan",
                    "cadinsan_n_com_PBF", "cadinsan_n_sem_PBF"]
    for column in nonnegative:
        values = base[column].dropna()
        if (values < 0).any() or not np.isclose(values % 1, 0).all():
            raise ValueError(f"{column}: contagem negativa ou não inteira.")
    for prefix, parts, total in (
        ("dai", ["altura_muito_baixa_n", "altura_baixa_n", "altura_adequada_n"], "avaliados_altura"),
    ):
        component_total = base[parts].sum(axis=1, min_count=len(parts))
        if not np.allclose(component_total[sv_rows], base.loc[sv_rows, total], equal_nan=False):
            raise ValueError(f"Soma das categorias incompatível com {total}.")
        base[f"{prefix}_n"] = base[parts[:2]].sum(axis=1, min_count=2)
        base[f"{prefix}_pct"] = base[f"{prefix}_n"].div(base[total].where(base[total] > 0)) * 100
        check_range(base[f"{prefix}_pct"], 0, 100, prefix)
    for scenario in ("com_PBF", "sem_PBF"):
        check_range(base[f"cadinsan_pct_{scenario}_arquivo"], 0, 100, "CadInsan no arquivo")
        denominator = base["cadastros_cadunico_cadinsan"]
        absolute = base[f"cadinsan_n_{scenario}"]
        if (absolute > denominator).fillna(False).any():
            raise ValueError("CadInsan absoluto excede o total de cadastros informado.")
        calculated = absolute.div(denominator.where(denominator > 0)) * 100
        reported = base[f"cadinsan_pct_{scenario}_arquivo"]
        complete = calculated.notna() & reported.notna()
        if not np.allclose(calculated[complete], reported[complete], atol=0.11):
            raise ValueError("Percentual CadInsan diverge da razão no arquivo.")
        base[f"cadinsan_pct_{scenario}"] = calculated
        base[f"cadinsan_diferenca_pp_{scenario}"] = calculated - reported
        check_range(calculated, 0, 100, "CadInsan recalculado")
    audit = pd.DataFrame([
        {"fonte": source, "registros_origem": len(table),
         "municipios_na_uniao": len(base), "correspondencias_sisvan": int((base[f"tem_{source}"] & sv_rows).sum()),
         "sisvan_sem_fonte": int((sv_rows & ~base[f"tem_{source}"]).sum()),
         "fonte_sem_sisvan": int((~sv_rows & base[f"tem_{source}"]).sum())}
        for source, table in tables.items()
    ])
    base = base.sort_values("codigo_ibge_6").reset_index(drop=True)
    return base, audit, hashes


def classify(base, quantile=0.75, min_evaluated=100, scenario="com_PBF"):
    """IVS/IDHM fixos; quantis nacionais nos demais; DAI exige denominador mínimo."""
    if not 0 < quantile < 1 or not isinstance(min_evaluated, int) or min_evaluated < 1:
        raise ValueError("Quantil deve estar entre 0 e 1 e o mínimo deve ser inteiro positivo.")
    if scenario not in ("com_PBF", "sem_PBF"):
        raise ValueError("Cenário deve ser com_PBF ou sem_PBF.")
    result = base.copy()
    result["cadinsan_pct"] = result[f"cadinsan_pct_{scenario}"]
    if f"cadinsan_pct_{scenario}_arquivo" in result:
        result["cadinsan_pct_arquivo"] = result[f"cadinsan_pct_{scenario}_arquivo"]
    result["cadinsan_n"] = result[f"cadinsan_n_{scenario}"]
    specs = [
        ("ivs", "ivs", pd.Series(True, index=result.index)),
        ("cadinsan", "cadinsan_pct", result["cadastros_cadunico_cadinsan"] > 0),
        ("dai", "dai_pct", result["avaliados_altura"] >= min_evaluated),
        ("idhm", "idhm", pd.Series(True, index=result.index)),
    ]
    thresholds = []
    for label, column, eligible in specs:
        valid = eligible & result[column].notna()
        if not valid.any():
            raise ValueError(f"Nenhum município elegível para {label}.")
        if label in FIXED_CUTS:
            cutoff, operator = FIXED_CUTS[label]
            method, percentile = "fixo", None
        else:
            cutoff = result.loc[valid, column].quantile(quantile)
            operator, method, percentile = ">=", "quantil", quantile * 100
        flag = pd.Series(pd.NA, index=result.index, dtype="boolean")
        flag.loc[valid] = (result.loc[valid, column] >= cutoff if operator == ">="
                           else result.loc[valid, column] < cutoff)
        result[f"criterio_{label}"] = flag
        thresholds.append({"indicador": label, "corte": float(cutoff),
                           "operador": operator, "metodo": method,
                           "municipios_referencia": int(valid.sum()),
                           "municipios_no_criterio": int(flag.fillna(False).sum()),
                           "percentil": percentile})
    primary = ["criterio_ivs", "criterio_idhm", "criterio_cadinsan", "criterio_dai"]
    result["elegivel_principal"] = result[primary].notna().all(axis=1)
    result["prioritario"] = result[primary].all(axis=1).where(result["elegivel_principal"]).astype("boolean")
    result["numero_criterios_primarios"] = result[primary].sum(axis=1).where(result["elegivel_principal"]).astype("Int64")
    social = (result["criterio_ivs"] & result["criterio_idhm"] & result["criterio_cadinsan"]).fillna(False)
    height = result["criterio_dai"].fillna(False)
    eligible = result["elegivel_principal"]
    groups = list(GROUP_COLORS)
    result["perfil"] = groups[4]
    result.loc[eligible, "perfil"] = groups[3]
    result.loc[eligible & height & ~social, "perfil"] = groups[2]
    result.loc[eligible & social & ~height, "perfil"] = groups[1]
    result.loc[eligible & social & height, "perfil"] = groups[0]
    reasons = []
    for row in result.itertuples():
        missing = []
        if pd.isna(row.ivs): missing.append("IVS ausente")
        if pd.isna(row.idhm): missing.append("IDHM ausente")
        if pd.isna(row.cadinsan_pct) or not row.cadastros_cadunico_cadinsan > 0:
            missing.append("CadInsan ausente ou denominador inválido")
        if pd.isna(row.avaliados_altura) or row.avaliados_altura <= 0:
            missing.append("sem avaliações de altura")
        elif row.avaliados_altura < min_evaluated:
            missing.append(f"menos de {min_evaluated} avaliações de altura")
        elif pd.isna(row.dai_pct): missing.append("DAI ausente")
        reasons.append("; ".join(missing))
    result["motivo_nao_classificacao"] = reasons
    result["cenario_cadinsan"] = scenario
    result["quantil_classificacao"] = quantile
    result["minimo_avaliados"] = min_evaluated
    cuts = pd.DataFrame(thresholds)
    cuts["percentil"] = cuts["percentil"].astype(object).where(cuts["percentil"].notna(), None)
    return result, cuts


def compare_rounding(base, reference, quantile=0.75, min_evaluated=100, scenario="com_PBF"):
    """Audita a diferença entre usar percentuais do CSV e razões sem arredondar."""
    rounded = base.copy()
    for option in ("com_PBF", "sem_PBF"):
        rounded[f"cadinsan_pct_{option}"] = rounded[f"cadinsan_pct_{option}_arquivo"]
    previous, thresholds = classify(rounded, quantile, min_evaluated, scenario)
    columns = ["codigo_ibge_6", "municipio", "uf", "cadinsan_pct", "cadinsan_pct_arquivo"]
    comparison = reference[columns].copy()
    old = previous.set_index("codigo_ibge_6")
    comparison["selecionado_percentual_csv"] = comparison["codigo_ibge_6"].map(old["prioritario"])
    comparison["selecionado_sem_arredondamento"] = reference["prioritario"].to_numpy()
    comparison["mudou_selecao"] = (comparison["selecionado_percentual_csv"].fillna(False).astype(bool)
                                   != comparison["selecionado_sem_arredondamento"].fillna(False).astype(bool))
    return comparison, thresholds


def sensitivity(base, reference, quantiles=(0.75, 0.80), minima=(30, 50, 100),
                scenarios=("com_PBF", "sem_PBF")):
    reference_set = set(reference.loc[reference["prioritario"].fillna(False), "codigo_ibge_6"])
    records, selections = [], []
    for scenario, quantile, minimum in product(scenarios, quantiles, minima):
        result, _ = classify(base, quantile, minimum, scenario)
        selected = set(result.loc[result["prioritario"].fillna(False), "codigo_ibge_6"])
        union = reference_set | selected
        records.append({"cenario": scenario, "quantil": quantile,
                        "corte_ivs": FIXED_CUTS["ivs"][0], "operador_ivs": FIXED_CUTS["ivs"][1],
                        "corte_idhm": FIXED_CUTS["idhm"][0], "operador_idhm": FIXED_CUTS["idhm"][1],
                        "minimo_avaliados": minimum,
                        "elegiveis": int(result["elegivel_principal"].sum()),
                        "prioritarios": len(selected), "coincidentes_principal": len(reference_set & selected),
                        "jaccard_principal": len(reference_set & selected) / len(union) if union else 1.0})
        selections.append(pd.DataFrame({"codigo_ibge_6": result["codigo_ibge_6"],
                                        "elegivel": result["elegivel_principal"],
                                        "selecionado": result["prioritario"].fillna(False).astype(int)}))
    all_selections = pd.concat(selections, ignore_index=True)
    stability = all_selections.groupby("codigo_ibge_6").agg(
        cenarios_elegiveis=("elegivel", "sum"), cenarios_selecionado=("selecionado", "sum"))
    stability["cenarios_testados"] = len(records)
    stability["fracao_cenarios_selecionado"] = stability["cenarios_selecionado"] / len(records)
    return pd.DataFrame(records), stability.reset_index()


def regional_summary(result):
    records = []
    for region, table in result.groupby("regiao", sort=True):
        record = {"regiao": region, "municipios": len(table),
                  "elegiveis": int(table["elegivel_principal"].sum()),
                  "prioritarios": int(table["prioritario"].fillna(False).sum())}
        record["pct_prioritarios_entre_elegiveis"] = (
            record["prioritarios"] / record["elegiveis"] * 100 if record["elegiveis"] else np.nan)
        for label, total in (("dai", "avaliados_altura"),):
            valid = table[total].gt(0) & table[f"{label}_n"].notna()
            denominator = table.loc[valid, total].sum()
            record[f"{label}_n"] = table.loc[valid, f"{label}_n"].sum()
            record[total] = denominator
            record[f"{label}_pct_agregado"] = (
                record[f"{label}_n"] / denominator * 100 if denominator else np.nan)
        records.append(record)
    return pd.DataFrame(records)


def dictionary():
    rows = [
        ("codigo_ibge_6", "Chave analítica por prefixo único", "texto", "Fontes; origem preservada", "Não implica harmonização histórica completa"),
        ("codigo_ibge_7", "Código informado nas fontes sociais", "texto", "IVS/CadÚnico/CadInsan", "Ausente quando nenhuma fonte social fornece o código"),
        ("ivs", "Índice de Vulnerabilidade Social", "0–1", "IVS, 2010", "Critério obrigatório: IVS ≥ 0,401, alta ou muito alta vulnerabilidade"),
        ("idhm", "Índice de Desenvolvimento Humano Municipal", "0–1", "IDHM, 2010", "Critério obrigatório: IDHM < 0,600, baixo ou muito baixo"),
        ("cadunico_valor_original", "Pessoas cadastradas, valor preservado do JSON", "pessoas", "CadÚnico, junho/2026", "Unidade e período descritos no catálogo com hash correspondente"),
        ("cadunico_pessoas_2026_06", "Alias explícito para pessoas cadastradas", "pessoas", "CadÚnico, junho/2026", "Contexto de demanda; não integra a regra principal"),
        ("cadunico_referencia", "Referência mensal do JSON", "ano-mês", "Catálogo Cozinhas Solidárias", "2026-06"),
        ("cadinsan_referencia", "Referência da base do CadInsan", "ano-mês", "Relatório oficial do MDS", "2025-01; não é uma média anual"),
        ("cadastros_cadunico_cadinsan", "Famílias consideradas no denominador do CSV", "famílias", "CadInsan, janeiro/2025", "Universo analisado; não equivale às pessoas do JSON"),
        ("cadinsan_pct", "100 × quantidade do cenário / famílias no denominador", "%", "CadInsan, janeiro/2025", "Sem arredondamento; cenários com/sem efeito do PBF"),
        ("cadinsan_pct_*_arquivo", "Percentuais do CSV preservados", "%", "CadInsan, janeiro/2025", "Arredondados; usados para comparação, não para seleção"),
        ("cadinsan_diferenca_pp_*", "Recalculado menos percentual do arquivo", "pontos percentuais", "Análise derivada", "Não confundir com variação percentual relativa"),
        ("cadinsan_n", "Famílias em risco estimado no cenário selecionado", "famílias", "CadInsan, janeiro/2025", "Não é contagem de pessoas nem medida direta de fome"),
        ("dai_n", "Altura muito baixa + altura baixa", "registros avaliados", "SISVAN, 2025", "Numerador recalculado"),
        ("dai_pct", "100 × dai_n / avaliados_altura", "%", "SISVAN, 2025", "Sem arredondamento; denominador zero gera NaN"),
        ("avaliados_altura", "Total avaliado em Altura X Idade", "registros avaliados", "SISVAN, 2025", "Não é medida de cobertura populacional"),
        ("*_valor_fonte", "Valores das células de contagem do XLSX", "texto", "SISVAN, 2025", "Preservados antes da interpretação da escala"),
        ("sisvan_escala_alterada", "Alguma contagem foi interpretada em escala diferente", "booleano", "Análise derivada", "A entrada não é reescrita"),
        ("sisvan_escalas_compativeis", "Número de soluções compatíveis com soma e percentuais", "inteiro", "Análise derivada", "Mais de uma: adotado o menor total compatível; ambiguidade da fonte"),
        ("criterio_*", "Flag para corte fixo (IVS/IDHM) ou quantil nacional (demais)", "booleano anulável", "Análise derivada", "Ausente quando indicador não é elegível; IVS ≥ 0,401 e IDHM < 0,600"),
        ("prioritario", "Coincidência de IVS, CadInsan e DAI elevados e IDHM baixo", "booleano anulável", "Análise derivada", "Quatro critérios obrigatórios; ausente para informação insuficiente"),
        ("numero_criterios_primarios", "Quantidade de critérios primários atendidos", "0–4", "Análise derivada", "Ausente quando falta informação para qualquer critério primário; não é ranking"),
        ("perfil", "Grupo exploratório de sobreposição", "categoria", "Análise derivada", "Não é classificação oficial"),
        ("fracao_cenarios_selecionado", "Fração das especificações que selecionam o município", "0–1", "Sensibilidade", "Não é probabilidade ou intervalo de confiança"),
    ]
    return pd.DataFrame(rows, columns=["variavel", "definicao", "unidade", "fonte_periodo", "observacao"])


def figure_style():
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.size": 10, "axes.spines.top": False,
                         "axes.spines.right": False, "figure.dpi": 110,
                         "savefig.dpi": 180})


def save_figure(fig, output, name):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    fig.savefig(output / f"{name}.png", bbox_inches="tight")
    fig.savefig(output / f"{name}.svg", bbox_inches="tight")


def distribution_figure(result, output):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    columns = [("ivs", "IVS — 2010"), ("idhm", "IDHM — 2010"),
               ("cadinsan_pct", "CadInsan (%) — cenário selecionado"),
               ("dai_pct", "DAI (%) — todos com avaliações"),
               ("avaliados_altura", "Avaliações de altura — escala log")]
    for ax, (column, title) in zip(axes.flat, columns):
        values = result[column].dropna()
        if column == "avaliados_altura":
            values = np.log10(values[values > 0])
            ax.set_xlabel("log10(total avaliado)")
        ax.hist(values, bins=35, color="#327c81", edgecolor="white", linewidth=0.4)
        ax.set_title(title)
        ax.set_ylabel("Municípios")
    axes.flat[-1].set_visible(False)
    fig.tight_layout()
    save_figure(fig, output, "01_distribuicoes")
    return fig


def association_figure(result, thresholds, output):
    import matplotlib.pyplot as plt
    cuts = thresholds.set_index("indicador")["corte"]
    valid = result[result["elegivel_principal"]]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    chosen = valid["prioritario"].fillna(False).to_numpy(dtype=bool)
    for ax, (column, label, criterion) in zip(axes, [
        ("ivs", "IVS (2010)", "ivs"), ("idhm", "IDHM (2010)", "idhm"),
        ("cadinsan_pct", "CadInsan (%)", "cadinsan")]):
        ax.scatter(valid[column], valid["dai_pct"], s=8, alpha=0.25, color="#327c81")
        ax.scatter(valid.loc[chosen, column], valid.loc[chosen, "dai_pct"],
                   s=13, alpha=0.7, color=GROUP_COLORS[list(GROUP_COLORS)[0]])
        ax.axhline(cuts["dai"], color="#777777", linestyle="--", linewidth=1)
        ax.axvline(cuts[criterion], color="#777777", linestyle="--", linewidth=1)
        ax.set(xlabel=label, ylabel="DAI (%)", title=f"{label} × DAI")
    fig.suptitle("Municípios elegíveis; vermelho = convergência dos quatro critérios")
    fig.tight_layout()
    save_figure(fig, output, "02_associacoes")
    return fig


def correlation_figure(result, output, minimum=100):
    import matplotlib.pyplot as plt
    cols = ["ivs", "idhm", "cadinsan_pct", "dai_pct"]
    data = result[cols].copy()
    data.loc[result["avaliados_altura"] < minimum, "dai_pct"] = np.nan
    corr = data.corr(method="spearman", min_periods=3)
    pair_counts = data.notna().astype(int).T @ data.notna().astype(int)
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(len(cols)), ["IVS", "IDHM", "CadInsan", "DAI"])
    ax.set_yticks(range(len(cols)), ["IVS", "IDHM", "CadInsan", "DAI"])
    for i in range(len(cols)):
        for j in range(len(cols)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}\nn={pair_counts.iloc[i, j]}",
                    ha="center", va="center", fontsize=9,
                    color="white" if abs(corr.iloc[i, j]) > 0.65 else "black")
    ax.set_title("Correlação de Spearman; pares disponíveis por comparação")
    fig.colorbar(im, ax=ax, shrink=0.8)
    fig.tight_layout()
    save_figure(fig, output, "03_correlacoes")
    return fig, corr, pair_counts


def sensitivity_figure(table, output):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(11, 4.5))
    labels = [f"{r.cenario}\nCadInsan/DAI P{r.quantil * 100:.0f}\nn≥{r.minimo_avaliados}"
              for r in table.itertuples()]
    ax.bar(range(len(table)), table["prioritarios"], color="#327c81")
    ax.set_xticks(range(len(table)), labels, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Municípios selecionados")
    ax.set_title("Sensibilidade: quantis, denominadores e cenários; IVS/IDHM com cortes fixos")
    fig.tight_layout()
    save_figure(fig, output, "04_sensibilidade")
    return fig


def geometry_audit(geojson, base):
    features = geojson["features"]
    codes_7 = pd.Series([str(f["properties"]["codarea"]) for f in features], dtype="string")
    codes(codes_7, 7, "Malha IBGE")
    prefixes = codes_7.str[:6]
    if prefixes.duplicated().any():
        raise ValueError("Malha com prefixos municipais duplicados.")
    existing = dict(zip(prefixes, codes_7))
    known = base["codigo_ibge_7"].notna()
    for row in base.loc[known].itertuples():
        if row.codigo_ibge_6 in existing and existing[row.codigo_ibge_6] != row.codigo_ibge_7:
            raise ValueError("Conflito de código municipal entre malha e bases sociais.")
    return pd.DataFrame({"codigo_ibge_6": base["codigo_ibge_6"],
                         "codigo_ibge_7_malha": base["codigo_ibge_6"].map(existing),
                         "tem_geometria": base["codigo_ibge_6"].isin(existing)})


def map_figure(geojson, result, column, title, output, filename, categories=None):
    """Mapa ilustrativo em coordenadas geográficas; anéis interiores preservados."""
    from textwrap import fill
    import matplotlib.pyplot as plt
    from matplotlib.collections import PatchCollection
    from matplotlib.colors import Normalize
    from matplotlib.path import Path as MplPath
    from matplotlib.patches import Patch, PathPatch
    lookup = result.set_index("codigo_ibge_6")[column].to_dict()
    patches, values, colors = [], [], []
    for feature in geojson["features"]:
        geometry = feature["geometry"]
        polygons = ([geometry["coordinates"]] if geometry["type"] == "Polygon"
                    else geometry["coordinates"])
        value = lookup.get(str(feature["properties"]["codarea"])[:6], np.nan)
        for polygon in polygons:
            vertices, commands = [], []
            for ring in polygon:
                coordinates = np.asarray(ring, dtype=float)[:, :2]
                if len(coordinates) < 4:
                    continue
                vertices.extend(coordinates)
                commands.extend([MplPath.MOVETO] + [MplPath.LINETO] * (len(coordinates) - 2) + [MplPath.CLOSEPOLY])
            if vertices:
                patches.append(PathPatch(MplPath(vertices, commands)))
                if categories is not None:
                    colors.append(categories.get(value, "#b9b9b9"))
                else:
                    values.append(float(value) if pd.notna(value) else np.nan)
    fig, ax = plt.subplots(figsize=(9, 9))
    collection = PatchCollection(patches, linewidths=0.08, edgecolors="#ffffff")
    if categories is not None:
        collection.set_facecolors(colors)
        ax.legend(handles=[Patch(facecolor=color, label=fill(label, width=42)) for label, color in categories.items()],
                  loc="lower left", fontsize=8, frameon=False)
    else:
        arr = np.ma.masked_invalid(values)
        collection.set_array(arr)
        cmap = plt.get_cmap("YlOrRd_r" if column == "idhm" else "YlOrRd").copy()
        cmap.set_bad("#b9b9b9")
        collection.set_cmap(cmap)
        finite = np.asarray(values)[np.isfinite(values)]
        if finite.size:
            collection.set_norm(Normalize(vmin=float(finite.min()), vmax=float(finite.max())))
        fig.colorbar(collection, ax=ax, fraction=0.035, pad=0.015)
        ax.legend(handles=[Patch(facecolor="#b9b9b9", label="Ausente / não elegível")],
                  loc="lower left", frameon=False)
    ax.add_collection(collection)
    ax.set(xlim=(-74.5, -32), ylim=(-34.5, 6), title=title)
    ax.set_aspect("equal")
    ax.axis("off")
    fig.text(0.15, 0.075, "IBGE: malha simplificada de apoio; coordenadas geográficas.\n"
             "Mapa ilustrativo; não mede cobertura ou concentração espacial estatística.", fontsize=8)
    save_figure(fig, output, filename)
    return fig

In [5]:
validacao_catalogo, proveniencia_catalogo = validate_catalogue(
    PASTA_DADOS / "documentacao/dataset_catalogue.json", PASTA_DADOS)
base, controle_integracao, hashes_entrada = prepare_base(PASTA_DADOS)
figure_style()
print("Municípios na união:", len(base))
print("Integridade das bases e correspondência com o catálogo verificadas.")
display(controle_integracao)
ausencias = base.loc[~base[["tem_ivs_idhm", "tem_cadunico", "tem_cadinsan", "tem_sisvan"]].all(axis=1),
                     ["codigo_ibge_6", "codigo_ibge_7", "municipio", "uf",
                      "tem_ivs_idhm", "tem_cadunico", "tem_cadinsan", "tem_sisvan"]]
display(ausencias)

Municípios na união: 5571
Integridade das bases e correspondência com o catálogo verificadas.


,fonte,registros_origem,municipios_na_uniao,correspondencias_sisvan,sisvan_sem_fonte,fonte_sem_sisvan
0,ivs_idhm,5565,5571,5565,6,0
1,cadunico,5564,5571,5564,7,0
2,cadinsan,5570,5571,5570,1,0
3,sisvan,5571,5571,5571,0,0


,codigo_ibge_6,codigo_ibge_7,municipio,uf,tem_ivs_idhm,tem_cadunico,tem_cadinsan,tem_sisvan
224,150475,1504752,MOJUI DOS CAMPOS,PA,False,False,True,True
803,220672,2206720,NAZARIA,PI,True,False,True,True
4503,421265,4212650,PESCARIA BRAVA,SC,False,False,True,True
4605,422000,4220000,BALNEARIO RINCAO,SC,False,False,True,True
4923,431454,4314548,PINTO BANDEIRA,RS,False,False,True,True
5160,500627,5006275,PARAISO DAS AGUAS,MS,False,False,True,True
5199,510183,<NA>,BOA ESPERANCA DO NORTE,MT,False,False,False,True


**Interpretação:** a união preserva municípios sem informações sociais.
O código SISVAN tem seis dígitos; as fontes sociais têm sete. A ligação usa
o prefixo único de seis dígitos, com verificação de conflitos. Isso não
demonstra que limites territoriais de 2010 e 2025 sejam
idênticos. CadÚnico JSON (pessoas, junho/2026) e `Cadastros_Cadunico`
(famílias, janeiro/2025) ficam separados. Sem denominador populacional
compatível, a contagem de pessoas não se transforma em proporção de cobertura.

## 4. Indicadores recalculados e qualidade

$$DAI(\%)=100\times\frac{N(\text{altura muito baixa})+N(\text{altura baixa})}
{N(\text{avaliados em altura por idade})}$$

$$CadInsan_{cenario}(\%)=100\times
\frac{Cadinsan\_absoluto\_{cenario}}{Cadastros\_Cadunico}$$

**DAI** é o percentual de déficit de altura para idade.
CadInsan usa famílias em risco estimado no
numerador e famílias do universo analisado no denominador.

Os percentuais são recalculados **sem arredondamento** para evitar empates
artificiais nos cortes; os valores de origem permanecem preservados.
Denominador zero gera percentual indefinido, não ausência de déficit.

**Leitura das contagens SISVAN:** o exportador apresenta algumas contagens
como números decimais (por exemplo, `1.02` para 1.020). O notebook interpreta
a escala por município, exigindo soma das três categorias igual ao total e
concordância com os percentuais oficiais (tolerância de 0,011 ponto percentual).
Falha de conciliação interrompe a análise. Quando mais de uma escala é
compatível, adota-se o menor total e registra-se a ambiguidade explicitamente.
Os valores da fonte e a auditoria são exportados; a entrada nunca é reescrita.
A ambiguidade pode afetar contagens e elegibilidade pelo mínimo de avaliações,
embora escalas uniformes não alterem o DAI percentual. Esta é uma limitação
do arquivo agregado, não uma identificação individual de crianças.

In [6]:
qualidade_sisvan = pd.DataFrame({
    "controle": ["Sem avaliações de altura", "Altura abaixo do mínimo principal",
                 "Contagens com escala interpretada", "Mais de uma escala compatível"],
    "municipios": [int(base["avaliados_altura"].eq(0).sum()),
                   int(base["avaliados_altura"].between(1, MINIMO_AVALIADOS - 1).sum()),
                   int(base["sisvan_escala_alterada"].fillna(False).sum()),
                   int(base["sisvan_escalas_compativeis"].gt(1).sum())]})
display(qualidade_sisvan)
colunas_auditoria = ["codigo_ibge_6", "municipio", "uf", "avaliados_altura",
                    "altura_muito_baixa_n", "altura_baixa_n", "altura_adequada_n",
                    "sisvan_escala_alterada", "sisvan_escalas_compativeis"]
colunas_auditoria += [c for c in base if c.endswith(("_valor_fonte", "_percentual_fonte"))]
interpretacao_sisvan = base.loc[base["tem_sisvan"], colunas_auditoria].copy()
display(base.loc[base["avaliados_altura"].eq(0),
                 ["municipio", "uf", "dai_pct", "avaliados_altura"]])
totais = []
for indicador, denominador in [("dai", "avaliados_altura")]:
    n = base[f"{indicador}_n"].sum()
    d = base[denominador].sum()
    totais.append({"indicador": indicador.upper(), "numerador": n,
                   "denominador": d, "percentual_agregado": n / d * 100})
display(pd.DataFrame(totais))

,controle,municipios
0,Sem avaliações de altura,1
1,Altura abaixo do mínimo principal,239
2,Contagens com escala interpretada,1733
3,Mais de uma escala compatível,3837


,municipio,uf,dai_pct,avaliados_altura
5199,BOA ESPERANCA DO NORTE,MT,NaN,0


,indicador,numerador,denominador,percentual_agregado
0,DAI,877708,7846413,11.186


**Interpretação:** esses totais descrevem os registros dos relatórios
consultados. Os percentuais agregados são razões entre somas, não
médias dos percentuais municipais. Muitos registros não garantem
representatividade. A população municipal de menores de cinco anos não está
disponível nesta análise; portanto não calculamos cobertura populacional.

## 5. Classificação exploratória

Ser prioritário exige IVS ≥ 0,401, IDHM < 0,600, CadInsan ≥ corte e DAI ≥ corte,
além de dados válidos para os quatro critérios e do mínimo de avaliações de
altura. IDHM ausente impede a classificação principal.
Informação insuficiente é diferente de ausência de risco.

In [7]:
classificados, cortes = classify(base, QUANTIL, MINIMO_AVALIADOS, CENARIO_CADINSAN)
display(cortes)
display(classificados["perfil"].value_counts().rename_axis("perfil").reset_index(name="municipios"))
sem_classificacao = classificados.loc[~classificados["elegivel_principal"],
    ["codigo_ibge_6", "municipio", "uf", "avaliados_altura", "motivo_nao_classificacao"]]
print("Municípios com informação insuficiente:", len(sem_classificacao))
comparacao_arredondamento, cortes_percentuais_csv = compare_rounding(
    base, classificados, QUANTIL, MINIMO_AVALIADOS, CENARIO_CADINSAN)
total_com_csv = int(comparacao_arredondamento["selecionado_percentual_csv"].fillna(False).sum())
total_recalculado = int(classificados["prioritario"].fillna(False).sum())
display(Markdown(f"**Efeito do arredondamento:** {total_com_csv} municípios com percentuais do CSV; "
                 f"{total_recalculado} com as razões sem arredondamento usadas na análise."))
display(comparacao_arredondamento.loc[comparacao_arredondamento["mudou_selecao"]])

,indicador,corte,operador,metodo,municipios_referencia,municipios_no_criterio,percentil
0,ivs,0.401,>=,fixo,5565,1981,None
1,cadinsan,10.970,>=,quantil,5570,1393,75.000
2,dai,12.381,>=,quantil,5331,1333,75.000
3,idhm,0.600,<,fixo,5565,1399,None


,perfil,municipios
0,Outras combinações,3655
1,DAI elevado sem convergência social-alimentar,1089
2,"IVS e CadInsan elevados, IDHM baixo, DAI abaix...",340
3,Informação insuficiente,245
4,Convergência dos quatro critérios,242


Municípios com informação insuficiente: 245


**Efeito do arredondamento:** 243 municípios com percentuais do CSV; 242 com as razões sem arredondamento usadas na análise.

,codigo_ibge_6,municipio,uf,cadinsan_pct,cadinsan_pct_arquivo,selecionado_percentual_csv,selecionado_sem_arredondamento,mudou_selecao
2122,292410,PEDRAO,BA,10.959,11.000,True,False,True


## 6. Distribuições, associações e redundância

As distribuições nutricionais mostram todos os municípios com avaliações.
Os gráficos de associação usam municípios elegíveis para a regra principal.
A correlação usa pares disponíveis, com denominadores nutricionais acima do
mínimo; cada par apresenta seu número de municípios. As linhas tracejadas
são os cortes adotados (fixos para IVS/IDHM, exploratórios para CadInsan/DAI).
Vermelho indica convergência dos quatro critérios.

In [8]:
display(distribution_figure(classificados, PASTA_FIGURAS))
plt.close("all")
display(association_figure(classificados, cortes, PASTA_FIGURAS))
plt.close("all")
figura_correlacao, correlacoes, n_pares = correlation_figure(classificados, PASTA_FIGURAS, MINIMO_AVALIADOS)
display(figura_correlacao)
plt.close("all")

<Figure size 1540x770 with 6 Axes>

<Figure size 1650x495 with 3 Axes>

<Figure size 770x660 with 2 Axes>

**Interpretação:** avaliar se valores desfavoráveis coincidem, a dispersão e
possíveis diferenças regionais. Correlações são descritivas: não demonstram
causalidade, relações individuais ou validação independente do CadInsan.
Este depende do CadÚnico; IVS e IDHM também podem conter informação redundante.
Não somamos os cinco indicadores em um índice.

## 7. Mapas nacionais e disponibilidade dos dados

Os mapas mostram os indicadores e sua coincidência no território municipal.
A correspondência dos códigos com a malha do IBGE é verificada; municípios
com informação insuficiente para a sobreposição aparecem em cinza.

In [9]:
caminho_malha = PASTA_DADOS / "apoio/ibge/malha_municipal_simplificada.geojson"
metadados_malha = json.loads((PASTA_DADOS / "apoio/ibge/malha_municipal_simplificada.metadados.json").read_text())
if hashlib.sha256(caminho_malha.read_bytes()).hexdigest() != metadados_malha["sha256"]:
    raise ValueError("Hash da malha diferente do registrado")
geometria = json.loads(caminho_malha.read_text())
controle_geometria = geometry_audit(geometria, base)
print("Municípios da base sem geometria:", int((~controle_geometria["tem_geometria"]).sum()))
print("Feições da malha sem registro na base:",
      len({str(f["properties"]["codarea"])[:6] for f in geometria["features"]} - set(base["codigo_ibge_6"])))
if GERAR_MAPAS:
    mapa = classificados.copy()
    mapa["disponibilidade"] = mapa["elegivel_principal"].map({True: "Elegível para regra", False: "Informação insuficiente"})
    display(map_figure(geometria, mapa, "disponibilidade", "Disponibilidade para a regra principal", PASTA_FIGURAS,
                       "05_disponibilidade", {"Elegível para regra": "#327c81", "Informação insuficiente": "#b9b9b9"}))
    plt.close("all")
    for coluna, titulo, nome in [("ivs", "IVS — contexto de 2010", "06_ivs"),
                                ("idhm", "IDHM — 2010; menor valor é mais desfavorável", "10_idhm"),
                                ("cadinsan_pct", f"CadInsan (%) — {CENARIO_CADINSAN}", "07_cadinsan"),
                                ("dai_pct", f"DAI (%) — altura com n ≥ {MINIMO_AVALIADOS}", "08_dai")]:
        if coluna == "dai_pct":
            mapa[coluna] = mapa[coluna].where(mapa["avaliados_altura"] >= MINIMO_AVALIADOS)
        display(map_figure(geometria, mapa, coluna, titulo, PASTA_FIGURAS, nome))
        plt.close("all")
    display(map_figure(geometria, classificados, "perfil", "Sobreposição de IVS, IDHM, CadInsan e DAI", PASTA_FIGURAS,
                       "09_sobreposicao", GROUP_COLORS))
    plt.close("all")

Municípios da base sem geometria: 0
Feições da malha sem registro na base: 0


<Figure size 990x990 with 1 Axes>

<Figure size 990x990 with 2 Axes>

<Figure size 990x990 with 2 Axes>

<Figure size 990x990 with 2 Axes>

<Figure size 990x990 with 2 Axes>

<Figure size 990x990 with 1 Axes>

**Interpretação:** o mapa final identifica coincidência municipal dos
critérios. Áreas visualmente próximas não constituem teste de agrupamento
espacial. A área dos polígonos não representa população ou quantidade de
famílias. O mapa de disponibilidade não é mapa de cobertura populacional.
No mapa do IDHM, valores menores recebem as cores mais vermelhas; nos
mapas de IVS, CadInsan e DAI, são os valores maiores.

## 8. Municípios selecionados e comparação regional

**Indicadores da seleção:** IVS, IDHM, CadInsan e DAI, com os denominadores
de famílias do CadInsan e avaliações de altura. As flags repetidas são omitidas.

**Caracterização dos municípios selecionados:** pessoas cadastradas no
CadÚnico em junho/2026 e famílias em risco estimado no cenário CadInsan
escolhido, de janeiro/2025. São unidades e períodos distintos, não somáveis.
As tabelas seguem a mesma ordem por UF e município, **sem ranking composto**.

**Comparação regional complementar:** DAI pela razão entre somas de
numeradores e denominadores de altura, usando todos os registros válidos,
não apenas os municípios selecionados.

In [10]:
prioritarios = classificados.loc[classificados["prioritario"].fillna(False)].sort_values(["uf", "municipio"])
colunas_selecao = {
    "municipio": "Município", "uf": "UF", "ivs": "IVS", "idhm": "IDHM",
    "cadinsan_pct": "CadInsan (%)", "dai_pct": "DAI (%)",
    "cadastros_cadunico_cadinsan": "Famílias no universo CadInsan",
    "avaliados_altura": "Avaliações de altura"}
colunas_caracterizacao = {
    "municipio": "Município", "uf": "UF",
    "cadunico_pessoas_2026_06": "Pessoas no CadÚnico — jun/2026",
    "cadinsan_n": "Famílias em risco estimado — jan/2025"}
indicadores_selecao = prioritarios[list(colunas_selecao)].rename(columns=colunas_selecao)
caracterizacao_municipios = prioritarios[list(colunas_caracterizacao)].rename(columns=colunas_caracterizacao)
print("Municípios com convergência:", len(prioritarios))
display(Markdown("### Indicadores da seleção"))
display(indicadores_selecao.head(30))
display(Markdown(f"### Caracterização dos municípios selecionados\n\nCenário CadInsan: `{CENARIO_CADINSAN}`."))
display(caracterizacao_municipios.head(30))
print("Prévia dos mesmos 30 municípios; a exportação mantém a lista completa e os nomes originais das colunas.")
resumo_regional = regional_summary(classificados)
display(Markdown("### Comparação regional — DAI"))
display(resumo_regional)

Municípios com convergência: 242


### Indicadores da seleção

,Município,UF,IVS,IDHM,CadInsan (%),DAI (%),Famílias no universo CadInsan,Avaliações de altura
53,ASSIS BRASIL,AC,0.500,0.588,14.877,26.857,1499.000,1158
62,MANOEL URBANO,AC,0.589,0.551,14.870,20.339,3006.000,1239
63,MARECHAL THAUMATURGO,AC,0.610,0.501,10.996,19.789,2410.000,1804
67,RODRIGUES ALVES,AC,0.553,0.567,13.608,16.371,3013.000,959
68,SANTA ROSA DO PURUS,AC,0.564,0.517,11.462,49.710,1012.000,690
1654,BARRA DE SANTO ANTONIO,AL,0.551,0.557,11.726,13.320,2891.000,1036
1668,CARNEIROS,AL,0.604,0.526,12.443,18.602,2210.000,758
1671,COLONIA LEOPOLDINA,AL,0.548,0.517,12.267,15.965,4133.000,1021
1672,COQUEIRO SECO,AL,0.530,0.586,14.702,15.423,857.000,402
1694,LAGOA DA CANOA,AL,0.490,0.552,13.851,12.412,3581.000,1418


### Caracterização dos municípios selecionados

Cenário CadInsan: `com_PBF`.

,Município,UF,Pessoas no CadÚnico — jun/2026,Famílias em risco estimado — jan/2025
53,ASSIS BRASIL,AC,9486.000,223.000
62,MANOEL URBANO,AC,11655.000,447.000
63,MARECHAL THAUMATURGO,AC,16086.000,265.000
67,RODRIGUES ALVES,AC,12891.000,410.000
68,SANTA ROSA DO PURUS,AC,6324.000,116.000
1654,BARRA DE SANTO ANTONIO,AL,13057.000,339.000
1668,CARNEIROS,AL,8010.000,275.000
1671,COLONIA LEOPOLDINA,AL,13508.000,507.000
1672,COQUEIRO SECO,AL,4046.000,126.000
1694,LAGOA DA CANOA,AL,15518.000,496.000


Prévia dos mesmos 30 municípios; a exportação mantém a lista completa e os nomes originais das colunas.


### Comparação regional — DAI

,regiao,municipios,elegiveis,prioritarios,pct_prioritarios_entre_elegiveis,dai_n,avaliados_altura,dai_pct_agregado
0,Centro-Oeste,468,440,2,0.455,67995,631574,10.766
1,Nordeste,1794,1784,147,8.240,297697,2568298,11.591
2,Norte,450,441,90,20.408,148137,1022788,14.484
3,Sudeste,1668,1596,3,0.188,268290,2600965,10.315
4,Sul,1191,1065,0,0.000,95589,1022788,9.346


## 9. Sensibilidade e estabilidade da seleção

São comparados percentis 75/80, mínimos de 30/50/100 avaliações e cenários
CadInsan com/sem efeito do Bolsa Família. IVS ≥ 0,401 e IDHM < 0,600 permanecem
fixos, e os quatro critérios continuam obrigatórios.

Cada combinação recalcula o corte do DAI entre municípios com o mínimo
escolhido: mudam tanto a elegibilidade quanto o universo do quantil.
O índice de **Jaccard** é a quantidade de municípios comuns às duas listas
dividida pela quantidade presente em pelo menos uma delas; 1 indica listas
idênticas e 0 indica nenhuma coincidência.

A frequência de seleção é a fração das especificações testadas, **não uma
probabilidade de risco ou medida de incerteza amostral**. Os cenários CadInsan
têm interpretações distintas e são discriminados na tabela.

In [11]:
quantis_teste = sorted(set(QUANTIS_SENSIBILIDADE + [QUANTIL]))
minimos_teste = sorted(set(MINIMOS_SENSIBILIDADE + [MINIMO_AVALIADOS]))
cenarios_teste = sorted(set(CENARIOS_SENSIBILIDADE + [CENARIO_CADINSAN]))
tabela_sensibilidade, estabilidade = sensitivity(base, classificados, quantis_teste, minimos_teste, cenarios_teste)
classificados = classificados.merge(estabilidade, on="codigo_ibge_6", validate="one_to_one")
prioritarios = classificados.loc[classificados["prioritario"].fillna(False)].sort_values(["uf", "municipio"])
display(tabela_sensibilidade)
display(sensitivity_figure(tabela_sensibilidade, PASTA_FIGURAS))
plt.close("all")
display(prioritarios[["municipio", "uf", "cenarios_elegiveis", "cenarios_selecionado",
                      "cenarios_testados", "fracao_cenarios_selecionado"]].head(30))

,cenario,quantil,corte_ivs,operador_ivs,corte_idhm,operador_idhm,minimo_avaliados,elegiveis,prioritarios,coincidentes_principal,jaccard_principal
0,com_PBF,0.750,0.401,>=,0.600,<,30,5555,243,242,0.996
1,com_PBF,0.750,0.401,>=,0.600,<,50,5531,243,242,0.996
2,com_PBF,0.750,0.401,>=,0.600,<,100,5326,242,242,1.000
3,com_PBF,0.800,0.401,>=,0.600,<,30,5555,179,179,0.740
4,com_PBF,0.800,0.401,>=,0.600,<,50,5531,179,179,0.740
5,com_PBF,0.800,0.401,>=,0.600,<,100,5326,179,179,0.740
6,sem_PBF,0.750,0.401,>=,0.600,<,30,5555,270,154,0.430
7,sem_PBF,0.750,0.401,>=,0.600,<,50,5531,270,154,0.430
8,sem_PBF,0.750,0.401,>=,0.600,<,100,5326,269,154,0.431
9,sem_PBF,0.800,0.401,>=,0.600,<,30,5555,189,116,0.368


<Figure size 1210x495 with 1 Axes>

,municipio,uf,cenarios_elegiveis,cenarios_selecionado,cenarios_testados,fracao_cenarios_selecionado
53,ASSIS BRASIL,AC,12,12,12,1.000
62,MANOEL URBANO,AC,12,12,12,1.000
63,MARECHAL THAUMATURGO,AC,12,9,12,0.750
67,RODRIGUES ALVES,AC,12,12,12,1.000
68,SANTA ROSA DO PURUS,AC,12,3,12,0.250
1654,BARRA DE SANTO ANTONIO,AL,12,3,12,0.250
1668,CARNEIROS,AL,12,9,12,0.750
1671,COLONIA LEOPOLDINA,AL,12,6,12,0.500
1672,COQUEIRO SECO,AL,12,9,12,0.750
1694,LAGOA DA CANOA,AL,12,3,12,0.250


## 10. Síntese para o laboratório e limitações

A síntese reúne a seleção principal, sua sensibilidade e os limites de
interpretação. As fontes não constituem um retrato simultâneo da população.

In [12]:
elegiveis = int(classificados["elegivel_principal"].sum())
selecionados = len(prioritarios)
resumo_execucao = f"""# Síntese exploratória\n\nForam preservados {len(base):,} municípios na união das fontes; \
{elegiveis:,} são elegíveis para a regra principal e {selecionados:,} apresentam convergência de IVS, \
CadInsan e DAI elevados e IDHM baixo ({selecionados / elegiveis * 100 if elegiveis else 0:.2f}% dos elegíveis).\n\n\
Parâmetros: IVS ≥ 0,401 e IDHM < 0,600 (fixos), percentil {QUANTIL * 100:.0f} \
para CadInsan/DAI, cenário `{CENARIO_CADINSAN}`, mínimo de \
{MINIMO_AVALIADOS} avaliações de altura. Quantis e mínimo são escolhas exploratórias; \
a sobreposição não é uma classificação oficial de risco alimentar.\n\n\
A seleção variou de {tabela_sensibilidade.prioritarios.min()} a \
{tabela_sensibilidade.prioritarios.max()} municípios nas especificações testadas. \
Consultar a tabela de sensibilidade para distinguir cenários, cortes e denominadores.\n\n\
**Referências:** IVS/IDHM de 2010; CadInsan: famílias de janeiro/2025; \
CadÚnico JSON: pessoas de junho/2026; SISVAN: acompanhamento de menores de cinco anos em 2025.\n\n\
**Limitações:** diferenças temporais e territoriais; SISVAN representa \
a população acompanhada; nenhum indicador de cobertura populacional foi calculado; \
pessoas do JSON e famílias do CadInsan não são unidades intercambiáveis; \
malha ilustrativa sem ano identificado pela API; associação \
municipal não demonstra causalidade nem relações individuais.\n\n\
**Processamento:** percentuais CadInsan recalculados sem arredondamento; \
percentuais originais preservados. Com os percentuais do CSV, esta configuração selecionaria \
{total_com_csv} municípios.\n"""
display(Markdown(resumo_execucao))

# Síntese exploratória

Foram preservados 5,571 municípios na união das fontes; 5,326 são elegíveis para a regra principal e 242 apresentam convergência de IVS, CadInsan e DAI elevados e IDHM baixo (4.54% dos elegíveis).

Parâmetros: IVS ≥ 0,401 e IDHM < 0,600 (fixos), percentil 75 para CadInsan/DAI, cenário `com_PBF`, mínimo de 100 avaliações de altura. Quantis e mínimo são escolhas exploratórias; a sobreposição não é uma classificação oficial de risco alimentar.

A seleção variou de 179 a 270 municípios nas especificações testadas. Consultar a tabela de sensibilidade para distinguir cenários, cortes e denominadores.

**Referências:** IVS/IDHM de 2010; CadInsan: famílias de janeiro/2025; CadÚnico JSON: pessoas de junho/2026; SISVAN: acompanhamento de menores de cinco anos em 2025.

**Limitações:** diferenças temporais e territoriais; SISVAN representa a população acompanhada; nenhum indicador de cobertura populacional foi calculado; pessoas do JSON e famílias do CadInsan não são unidades intercambiáveis; malha ilustrativa sem ano identificado pela API; associação municipal não demonstra causalidade nem relações individuais.

**Processamento:** percentuais CadInsan recalculados sem arredondamento; percentuais originais preservados. Com os percentuais do CSV, esta configuração selecionaria 243 municípios.


## 11. Resultados para consulta e reprodução

Exportamos a base derivada, lista completa de municípios com convergência,
cortes, controles de qualidade, sensibilidade, dicionário, resumo e figuras
em um ZIP. O registro de execução conserva parâmetros e identificação das
entradas para reprodução. A exportação não inclui dados pessoais individuais.
O ZIP pode ser baixado pela aba de arquivos do Colab ou pela opção de
download nos parâmetros.

In [13]:
identificador = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
destino = PASTA_SAIDA / f"execucao_{identificador}"
destino.mkdir()
tabelas = {"base_analitica": classificados, "municipios_prioritarios": prioritarios,
           "cortes": cortes, "controle_integracao": controle_integracao,
           "municipios_sem_correspondencia": ausencias, "sem_classificacao": sem_classificacao,
           "qualidade_sisvan": qualidade_sisvan, "interpretacao_sisvan": interpretacao_sisvan,
           "controle_geometria": controle_geometria,
           "resumo_regional": resumo_regional, "sensibilidade": tabela_sensibilidade,
           "estabilidade": estabilidade, "dicionario_variaveis": dictionary(),
           "validacao_catalogo": validacao_catalogo,
           "comparacao_arredondamento": comparacao_arredondamento,
           "cortes_percentuais_csv": cortes_percentuais_csv,
           "correlacoes_spearman": correlacoes.rename_axis("indicador").reset_index(),
           "n_pares_correlacao": n_pares.rename_axis("indicador").reset_index()}
for nome, tabela in tabelas.items():
    tabela.to_csv(destino / f"{nome}.csv", index=False, encoding="utf-8-sig")
figuras_desta_execucao = ["01_distribuicoes", "02_associacoes", "03_correlacoes", "04_sensibilidade"]
if GERAR_MAPAS:
    figuras_desta_execucao += ["05_disponibilidade", "06_ivs", "07_cadinsan", "08_dai", "09_sobreposicao", "10_idhm"]
(destino / "figuras").mkdir()
for nome in figuras_desta_execucao:
    for extensao in ["png", "svg"]:
        shutil.copy2(PASTA_FIGURAS / f"{nome}.{extensao}", destino / "figuras")
ambiente = {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
            "matplotlib": matplotlib.__version__, "scipy": scipy.__version__}
manifesto = {"executado_em": datetime.now(timezone.utc).isoformat(),
             "proveniencia_sisvan": sisvan_metadata(PASTA_DADOS),
             "formula_dai_pct": "100 * (altura_muito_baixa_n + altura_baixa_n) / avaliados_altura; total zero gera NaN",
             "quantil": QUANTIL, "minimo_avaliados": MINIMO_AVALIADOS,
             "cenario_cadinsan": CENARIO_CADINSAN, "cortes": cortes.to_dict("records"),
             "proveniencia_catalogo": proveniencia_catalogo,
             "formula_cadinsan_pct": "100 * Cadinsan_absoluto_cenario / Cadastros_Cadunico; sem arredondamento",
             "municipios_selecionados_com_percentuais_csv": total_com_csv,
             "municipios_selecionados_sem_arredondamento": total_recalculado,
             "sensibilidade": tabela_sensibilidade.to_dict("records"), "ambiente": ambiente,
             "hashes_entrada": hashes_entrada.to_dict("records"), "malha": metadados_malha,
             "codigo_analise_sha256": CODIGO_ANALISE_SHA256,
             "regra": "IVS >= 0.401 e IDHM < 0.600 e CadInsan >= corte e DAI >= corte; dados e denominadores válidos",
             "criterios_primarios": ["ivs", "idhm", "cadinsan", "dai"],
             "cortes_sociais_fixos": {indicador: {"corte": valor, "operador": operador}
                                     for indicador, (valor, operador) in FIXED_CUTS.items()},
             "indicadores_com_quantil": ["cadinsan", "dai"],
             "limites": ["Estudo ecológico exploratório", "Sem inferência causal",
                         "Sem cobertura populacional calculada", "Referências temporais distintas",
                         "Pessoas CadÚnico de junho/2026 e famílias CadInsan de janeiro/2025 não são intercambiáveis"]}
(destino / "manifesto_execucao.json").write_text(json.dumps(manifesto, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
(destino / "resumo.md").write_text(resumo_execucao, encoding="utf-8")
shutil.copytree(PASTA_DADOS / "documentacao", destino / "documentacao_entradas")
shutil.copy2(PASTA_DADOS / "README.md", destino / "documentacao_entradas/README_SELECAO.md")
shutil.copy2(PASTA_DADOS / "SHA256SUMS", destino / "documentacao_entradas")
arquivo_zip = shutil.make_archive(str(destino), "zip", root_dir=destino)
print("ZIP pronto:", arquivo_zip)
if BAIXAR_RESULTADOS_NO_COLAB:
    try:
        from google.colab import files
    except ImportError:
        print("Fora do Colab: utilizar o ZIP no caminho informado.")
    else:
        files.download(arquivo_zip)

ZIP pronto: /tmp/risco-notebook-altura-xcuhiefx/resultados_sobreposicao/resultados/execucao_20261004T210928132724Z.zip


## Referências

- [Repositório de Cozinhas Solidárias — procedência das bases sociais](https://github.com/TriangulosTecnologia/cozsolidarias).
- [Atlas do Desenvolvimento Humano — PNUD, Ipea e FJP](https://www.undp.org/pt/brazil/desenvolvimento-humano/atlas-do-desenvolvimento-humano-no-brasil).
- [Cadastro Único — fonte SAGI/MDS](https://aplicacoes.mds.gov.br/sagi/servicos/misocial).
- [CadInsan — MDS](https://www.gov.br/mds/pt-br/Sisan/vigilancia-do-sisan/CADinsan).
- [Relatório CadInsan com referência janeiro/2025 — MDS](https://www.gov.br/mds/pt-br/Sisan/vigilancia-do-sisan/CADINSAN2025.pdf).
- [SISVAN — Ministério da Saúde](https://www.gov.br/saude/pt-br/composicao/saps/vigilancia-alimentar-e-nutricional/sisvan).
- [IVS e IDHM — Ipea](https://repositorio.ipea.gov.br/bitstream/11058/8257/2/vulnerability.pdf).
- [Faixas do IVS — Atlas do Ipea](https://repositorio.ipea.gov.br/bitstream/11058/4381/1/Atlas_da_vulnerabilidade_social_nos_municipios_brasileiros.pdf).
- [Faixas do IDHM — PNUD](https://www.undp.org/sites/g/files/zskgke326/files/2024-05/anexo_estatistico_pnud_21maio24_isbn_web2.pdf).
- [API de malhas simplificadas v4 — IBGE](https://servicodados.ibge.gov.br/api/docs/malhas?versao=4).